# Lab 1: Primes and Euclid leave trails
Use the sieve, Euclid's algorithm, and continued fractions to expose structure; then certify each pattern by hand.

## The sieve of Eratosthenes
Record not only which numbers are composite, but which prime first crosses each one out.

In [ ]:
from math import isqrt

def sieve_trace(N):
    alive = [True] * (N + 1)
    alive[0] = alive[1] = False
    crossed_by = {}
    for p in range(2, isqrt(N) + 1):
        if alive[p]:
            for m in range(p*p, N + 1, p):
                if alive[m]:
                    crossed_by[m] = p
                alive[m] = False
    primes = [n for n in range(2, N + 1) if alive[n]]
    return primes, crossed_by

primes, crossed_by = sieve_trace(100)
primes

**Prove the stopping rule.** Why must a composite $n\le N$ have a prime factor at most $\sqrt N$? Why is it safe to begin crossing out multiples of $p$ at $p^2$?

In [ ]:
def euclid_trace(a, b):
    rows = []
    while b:
        q, r = divmod(a, b)
        rows.append((a, q, b, r))
        a, b = b, r
    return rows, a

rows, d = euclid_trace(391, 299)
rows, d

**Task.** Explain why replacing `(a,b)` by `(b,r)` preserves every common divisor. Then use the trace to reconstruct Bezout coefficients.

In [ ]:
d, x, y = xgcd(391, 299)
d, x, y, 391*x + 299*y

## Continued fractions
Compare Euclid's quotients with the partial quotients below.

In [ ]:
q = QQ(415)/93
cf = continued_fraction(q)
cf, list(cf.convergents())

## Convergents are extended-Euclidean coefficients
For $A/B=[a_0;\ldots,a_n]$, compare the Euclidean remainder $r_k$ with $(-1)^k(q_kA-p_kB)$.

In [ ]:
A, B = 415, 93
rows, d = euclid_trace(A, B)
convs = list(continued_fraction(QQ(A)/B).convergents())
comparison = []
for k, c in enumerate(convs):
    p, q = c.numerator(), c.denominator()
    remainder_from_cf = (-1)^k * (q*A - p*B)
    comparison.append((k, rows[k][1], c, rows[k][3], remainder_from_cf))
comparison   # (index, partial quotient, convergent, Euclid remainder, CF formula)

**Task.** Prove the remainder formula from the two recurrences. Explain why the penultimate convergent supplies Bezout coefficients for the gcd.

In [ ]:
alpha = continued_fraction(sqrt(2))
conv = [alpha.convergent(k) for k in range(10)]
[(c, c.numerator()^2 - 2*c.denominator()^2) for c in conv]

In [ ]:
lower = conv[0::2]
upper = conv[1::2]
lower_increases = all(lower[i] < lower[i+1] for i in range(len(lower)-1))
upper_decreases = all(upper[i] > upper[i+1] for i in range(len(upper)-1))
[(k, c, 'below' if c < sqrt(2) else 'above') for k, c in enumerate(conv)], lower_increases, upper_decreases

**The Mobius derivative trick.** With $M_k(t)=(p_kt+p_{k-1})/(q_kt+q_{k-1})$, compute $M_k'(t)$. Use its alternating sign to prove that the zero-indexed even convergents increase below $x$, while the odd convergents decrease above $x$.

**Conjecture and prove.** Which indices solve $p^2-2q^2=1$, and which solve $-1$? Use the recurrence, not the decimal approximations.

**Extension.** Replace 2 by nonsquare values from 3 through 20. Record the repeating block and the first solution to $p^2-dq^2=1$.

## The matrix engine
Multiply the digit matrices $D(a)=\begin{pmatrix}a&1\\1&0\end{pmatrix}$. The two columns should be consecutive convergents, and the determinant should alternate between $1$ and $-1$.

In [ ]:
def convergent_matrix(coeffs):
    C = identity_matrix(ZZ, 2)
    for a in coeffs:
        C *= matrix(ZZ, [[a, 1], [1, 0]])
    return C

C = convergent_matrix([4, 2, 6, 7])
C, C.det(), C.inverse()

**Explain.** Why does the inverse still have integer entries? Locate the Bezout identity $13(415)-58(93)=1$ in the displayed matrices.

## Repeating base expansions are finite-state processes
Long division has only finitely many possible remainders. The first repeated remainder determines the repeating block.

In [ ]:
def base_expansion_trace(m, q, b=10, steps=30):
    remainder = m % q
    seen = {}
    digits = []
    trace = []
    for n in range(steps):
        if remainder in seen:
            return digits, trace, seen[remainder], n - seen[remainder]
        seen[remainder] = n
        digit, new_remainder = divmod(b*remainder, q)
        trace.append((n, remainder, digit, new_remainder))
        digits.append(digit)
        remainder = new_remainder
    return digits, trace, None, None

base_expansion_trace(1, 7), base_expansion_trace(5, 12)

**Compare two periodicities.** Decimal repetition comes from finitely many remainders. Continued-fraction repetition for a quadratic irrational comes from finitely many complete-quotient states. Verify Sage's expansions below, then use the matrix of one repeating block to recover a quadratic equation.

In [ ]:
continued_fraction(sqrt(2)), continued_fraction(1 + sqrt(2)), continued_fraction((1 + sqrt(5))/2)

## Periodicity meets Pell
A solution $u^2-Dv^2=1$ makes $U=\begin{pmatrix}u&Dv\\v&u\end{pmatrix}$ an $SL(2,\mathbb Z)$ matrix. Its powers multiply by powers of $u+v\sqrt D$ and generate further Pell solutions.

In [ ]:
D = 3
U = matrix(ZZ, [[2, 3], [1, 2]])
pell_rows = []
for n in range(1, 7):
    p, q = (U^n).column(0)
    pell_rows.append((n, p, q, p^2 - D*q^2, abs(sqrt(D) - QQ(p)/q), QQ(1)/(2*q^2)))
pell_rows   # the last two columns verify Legendre's strict inequality

**Prove both arrows.** Factor $|D-p^2/q^2|$ to show why every $\pm1$ Pell solution satisfies Legendre's bound. Then explain how the repeating period of $\sqrt D$ supplies the first generator matrix.

## Exact Pell remainders and palindromic periods
These cells accompany Chapter 3's Pell section and Exercises 27–31. They use only standard Python integer arithmetic and can be run independently of the earlier Sage cells. No decimal approximation to a square root is used.

The book proves
\[
p_k^2-Dq_k^2=(-1)^{k+1}Q_{k+1},\qquad Q_n=1\Longleftrightarrow L\mid n\quad(n\ge1).
\]
The period ends in $2a_0$, and the digits before it form a palindrome. The computation below detects a repeated **state**, without assuming that the final digit is $2a_0$. This lets us check the theorem's predictions separately.

In [ ]:
from math import isqrt, gcd
from functools import reduce


def sqrt_cf_states(D):
    D = int(D)
    if D < 2 or isqrt(D)**2 == D:
        raise ValueError("D must be a nonsquare integer at least 2")
    a0 = isqrt(D)
    P, Q, a = 0, 1, a0
    seen, states = {}, []
    while True:
        P = a*Q - P
        numerator = D - P*P
        assert numerator > 0 and numerator % Q == 0
        Q = numerator // Q
        a = (a0 + P) // Q
        state = (P, Q)
        if state in seen:
            assert seen[state] == 0  # repetition starts at the first tail
            return a0, states
        seen[state] = len(states)
        states.append((P, Q, a))


def pell_convergents(D, count):
    D = int(D)
    if D < 2 or isqrt(D)**2 == D:
        raise ValueError("D must be a nonsquare integer at least 2")
    a0 = isqrt(D)
    P, Q, a = 0, 1, a0
    p_old, p, q_old, q = 0, 1, 1, 0
    rows = []
    for k in range(count):
        p_old, p = p, a*p + p_old
        q_old, q = q, a*q + q_old
        next_P = a*Q - P
        numerator = D - next_P*next_P
        assert numerator > 0 and numerator % Q == 0
        next_Q = numerator // Q
        rows.append((k, a, p, q, next_Q, p*p - D*q*q))
        P, Q = next_P, next_Q
        a = (a0 + P) // Q
    return rows


def fundamental_pell(D):
    a0, states = sqrt_cf_states(D)
    L = len(states)
    index = L - 1 if L % 2 == 0 else 2*L - 1
    rows = pell_convergents(D, index + 1)
    assert rows[-1][-1] == 1
    assert all(row[-1] != 1 for row in rows[:-1])
    return rows[-1][2:4]

In [ ]:
D = 13
rows = pell_convergents(D, 10)
print("k  a_k  p_k  q_k  Q_(k+1)  Pell remainder")
for row in rows:
    k, a, p, q, Q_next, remainder = row
    assert remainder == (-1)**(k + 1)*Q_next
    print(*row)
assert rows[4][2:4] == (18, 5)
assert rows[9][2:4] == (649, 180)

**Explain the index.** Why do $18/5$ and $649/180$ occur immediately before an occurrence of the final digit $6$? Explain why the error estimate alone cannot establish that the Pell remainder is $\pm1$.

For each row below, identify the palindrome and predict whether the negative equation is solvable. Then connect each assertion in the code to a proved statement in the book. Checking finitely many examples supports the exposition; it does not replace those proofs.

In [ ]:
for D in (2, 3, 5, 6, 7, 13, 14, 23, 61):
    a0, states = sqrt_cf_states(D)
    period = [state[2] for state in states]
    L = len(period)
    assert period[-1] == 2*a0
    assert period[:-1] == period[-2::-1]
    for k, a, p, q, Q_next, remainder in pell_convergents(D, 4*L):
        assert remainder == (-1)**(k + 1)*Q_next
        assert (Q_next == 1) == ((k + 1) % L == 0)
    print(D, "period", period, "first +1 solution", fundamental_pell(D))
assert fundamental_pell(61) == (1766319049, 226153980)

## Archimedes' Cattle of the Sun
The problem's [ancient conditions](https://math.nyu.edu/Archimedes/Cattle/Statement.html) give seven linear equations, a square condition, and a triangular condition. See Ilan Vardi, [“Archimedes' Cattle Problem,” *American Mathematical Monthly* 105 (1998), 305–319](https://doi.org/10.1080/00029890.1998.12004887), and the derivation in the book.

The order of entries below is white, black, yellow, and dappled bulls, followed by cows in the same color order. First check all seven proportions, the primitive gcd, and the total. The linear solution alone does not satisfy the square condition.

In [ ]:
primitive_herd = (10366482, 7460514, 4149387, 7358060,
                  7206360, 4893246, 5439213, 3515820)


def check_cattle_proportions(herd):
    W, B, Y, S, w, b, y, s = herd
    return all((6*W == 5*B + 6*Y,
                20*B == 9*S + 20*Y,
                42*S == 13*W + 42*Y,
                12*w == 7*(B + b),
                20*b == 9*(S + s),
                42*y == 13*(W + w),
                30*s == 11*(Y + y)))


assert check_cattle_proportions(primitive_herd)
assert reduce(gcd, primitive_herd) == 1
assert sum(primitive_herd) == 50389082
squarefree_scale = 3*11*29*4657
assert squarefree_scale == 4456749
assert all(4657 % d for d in range(2, isqrt(4657) + 1))
assert primitive_herd[0] + primitive_herd[1] == 4*squarefree_scale
triangle_coefficient = (primitive_herd[2] + primitive_herd[3])*squarefree_scale
assert 8*triangle_coefficient == 410286423278424 == 4729494*9314**2
print("The seven proportions and both reduction constants are verified.")

The reduced equation is $X^2-4729494 Z^2=1$, **with $9314\mid Z$**. Compute its least period and fundamental solution, then search for the first admissible power modulo $9314$. The second coefficient must vanish; there is no requirement that the first coefficient be $+1$ rather than $-1$ modulo $9314$.

In [ ]:
cattle_D = 4729494
cattle_a0, cattle_states = sqrt_cf_states(cattle_D)
cattle_period = [state[2] for state in cattle_states]
assert cattle_a0 == 2174 and len(cattle_period) == 92
assert cattle_period[-1] == 4348
assert cattle_period[:-1] == cattle_period[-2::-1]
cattle_u, cattle_v = fundamental_pell(cattle_D)
assert cattle_u == 109931986732829734979866232821433543901088049
assert cattle_v == 50549485234315033074477819735540408986340
assert cattle_u*cattle_u - cattle_D*cattle_v*cattle_v == 1

modulus = 9314
u_mod, v_mod, D_mod = cattle_u % modulus, cattle_v % modulus, cattle_D % modulus
assert (u_mod, v_mod, D_mod) == (9063, 7708, 7296)
a, b = 1, 0
first_admissible = None
for j in range(1, 2330):
    a, b = ((u_mod*a + D_mod*v_mod*b) % modulus,
            (v_mod*a + u_mod*b) % modulus)
    if b == 0:
        first_admissible = j
        break
assert first_admissible == 2329
assert (a, b) == (9313, 0)
print("Period length:", len(cattle_period))
print("Fundamental Pell solution:", (cattle_u, cattle_v))
print("First admissible power and residues:", first_admissible, (a, b))

**Why is this minimal?** Use the theorem that every positive Pell solution is a power of the fundamental solution, and the fact that its second coefficient increases. Prove that all later admissible exponents are multiples of $2329$ (Exercise 34).

We can now form the exact solution by repeated squaring. The next cell does not print hundreds of thousands of digits: it verifies all the herd's conditions, certifies the decimal length using exact inequalities, and displays only its first and last 20 digits. No floating-point arithmetic or global integer-string conversion setting is needed.

In [ ]:
def multiply_surd_pairs(D, left, right):
    a, b = left
    c, d = right
    return a*c + D*b*d, a*d + b*c


def power_surd_pair(D, pair, exponent):
    D, exponent = int(D), int(exponent)
    if exponent < 0:
        raise ValueError("Use a nonnegative exponent")
    result = (1, 0)
    base = tuple(map(int, pair))
    while exponent:
        if exponent % 2:
            result = multiply_surd_pairs(D, result, base)
        exponent //= 2
        if exponent:
            base = multiply_surd_pairs(D, base, base)
    return result


X, Z = power_surd_pair(cattle_D, (cattle_u, cattle_v), first_admissible)
assert X*X - cattle_D*Z*Z == 1
assert X % 2 == 1 and Z % 9314 == 0
r = Z // 9314
t = squarefree_scale*r*r
herd = tuple(coefficient*t for coefficient in primitive_herd)
assert all(count > 0 for count in herd)
assert check_cattle_proportions(herd)
W, B, Y, S, w, b, y, s = herd
h = (X - 1) // 2
assert W + B == (2*squarefree_scale*r)**2
assert Y + S == h*(h + 1)//2
total = sum(herd)
assert total == 50389082*4456749*r*r
assert 10**206544 <= total < 10**206545
first_twenty = total // 10**(206545 - 20)
last_twenty = str(total % 10**20).zfill(20)
assert first_twenty == 77602714064868182695
print({"decimal_digits": 206545,
       "first_20_digits": first_twenty,
       "last_20_digits": last_twenty,
       "all_cattle_conditions_verified": True})

## A Khinchin--Levy experiment with $\pi$ and $e$
For $x=[a_0;a_1,a_2,\ldots]$, compute $G_n=(a_1\cdots a_n)^{1/n}$ and $B_n=q_n^{2/n}$. The almost-everywhere targets are about $2.685452$ and $10.731016$. Numerical agreement for a named constant is evidence, not proof.

In [ ]:
def khinchin_levy_rows(x, cutoffs):
    R = RealField(80)
    cf = continued_fraction(x)
    q_old, q_new = ZZ(1), ZZ(0)
    sum_log = R(0)
    rows = []
    for k in range(max(cutoffs) + 1):
        a = ZZ(cf[k])
        q_old, q_new = q_new, a*q_new + q_old
        if k > 0:
            sum_log += R(a).log()
        if k in cutoffs:
            G = (sum_log/k).exp()
            B = (2*R(q_new).log()/k).exp()
            rows.append((k, G, B, B.log()/R(10).log()))
    return rows

pi_rows = khinchin_levy_rows(pi, [100, 1000, 10000])
pi_rows   # n, G_n(pi), B_n(pi), decimal digits per CF term

**Why $e$ is different.** Euler's expansion is $e=[2;1,2,1,1,4,1,1,6,\ldots]$. For $n=3m$, the product of the noninitial partial quotients is exactly $2^m m!$, so $G_{3m}(e)\sim(2m/e)^{1/3}\to\infty$. The next cell displays this provable divergence.

In [ ]:
R = RealField(80)
[(3*m, ((m*R(2).log() + R(factorial(m)).log())/(3*m)).exp())
 for m in [10, 100, 1000, 10000]]

## Gosper's streaming arithmetic
A homographic state $M$ represents $(At+B)/(Ct+D)$. Absorbing an input digit multiplies $M$ on the right by its digit matrix; emitting a certified output digit multiplies on the left by $\begin{pmatrix}0&1\\1&-k\end{pmatrix}$. The following exact-rational calculation reproduces the first four output digits of $\pi/2$.

In [ ]:
def digit_matrix(m):
    return matrix(ZZ, [[m, 1], [1, 0]])

def absorb(M, m):
    return M * digit_matrix(m)

def emit(M, k):
    return matrix(ZZ, [[0, 1], [1, -k]]) * M

def mobius_endpoint_range(M):
    A, B, C, D = M.list()
    values = [QQ(A + B)/QQ(C + D), QQ(A)/QQ(C)]
    return min(values), max(values)

M = matrix(ZZ, [[1, 0], [0, 2]])       # x/2
M = absorb(absorb(M, 3), 7)            # pi begins [3;7,...]
rows = []
for _ in range(4):
    lo, hi = mobius_endpoint_range(M)
    k = floor(lo)
    assert floor(hi) == k               # sufficient in these four rows
    rows.append((k, M, lo, hi))
    M = emit(M, k)
rows, M                                 # digits 1,1,1,3; then state 2t

Now absorb the next two input digits, 15 and 1. The two endpoint values are 31 and 32, but the value 32 occurs only at the nonattained limit $t=\infty$. Thus the exact range is $[31,32)$ and the next output digit is 31. This is why a full implementation must retain open and closed endpoint information.

In [ ]:
M = absorb(absorb(M, 15), 1)
mobius_endpoint_range(M), M             # [31,32), so emit 31